# Delta SOH Hybrid Model Comparison

This notebook compares SOH prediction models with `--target-mode delta`, so each neural model predicts future SOH change rather than absolute SOH.

It supports both local Jupyter and Google Colab.

For Colab, use one of these data setup options:

1. Put the project folder at `Google Drive/MyDrive/UROP/` with this structure:

```text
MyDrive/UROP/
  compare_soh_models.py
  soh_gru_dsconv_pipeline.py
  raw_samples/
    *.pkl
```

2. Or upload a zip containing `compare_soh_models.py`, `soh_gru_dsconv_pipeline.py`, and `raw_samples/` when prompted.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
DATA_DIR = "raw_samples"

if IN_COLAB:
    from google.colab import drive, files

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/UROP")
    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/UROP")

    if DRIVE_PROJECT_DIR.exists():
        print(f"Copying project from Drive: {DRIVE_PROJECT_DIR}")
        if PROJECT_DIR.exists():
            shutil.rmtree(PROJECT_DIR)
        shutil.copytree(DRIVE_PROJECT_DIR, PROJECT_DIR)
    else:
        print("Drive project folder not found at /content/drive/MyDrive/UROP")
        print("Upload a zip containing compare_soh_models.py, soh_gru_dsconv_pipeline.py, and raw_samples/.")
        uploaded = files.upload()
        zip_paths = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
        if not zip_paths:
            raise FileNotFoundError("No zip file uploaded. Upload a project zip or create Drive/MyDrive/UROP.")
        extract_root = Path("/content/UROP_upload")
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_paths[0]) as zf:
            zf.extractall(extract_root)
        candidates = list(extract_root.rglob("compare_soh_models.py"))
        if not candidates:
            raise FileNotFoundError("Uploaded zip does not contain compare_soh_models.py")
        PROJECT_DIR = candidates[0].parent
else:
    local_candidates = [
        Path.cwd(),
        Path(r"C:\Users\kyucho\Documents\UROP"),
    ]
    PROJECT_DIR = next(
        (p for p in local_candidates if (p / "compare_soh_models.py").exists()),
        local_candidates[0],
    )

SCRIPT = PROJECT_DIR / "compare_soh_models.py"
DATA_PATH = PROJECT_DIR / DATA_DIR

print("Colab:", IN_COLAB)
print("Project:", PROJECT_DIR)
print("Script exists:", SCRIPT.exists())
print("Data exists:", DATA_PATH.exists())
if not SCRIPT.exists():
    raise FileNotFoundError(f"Missing script: {SCRIPT}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing data directory: {DATA_PATH}")
print("pkl files:", len(list(DATA_PATH.rglob("*.pkl"))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying project from Drive: /content/drive/MyDrive/UROP
Colab: True
Project: /content/UROP
Script exists: True
Data exists: True
pkl files: 18


In [2]:
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
    elif IN_COLAB:
        print("Colab GPU is not enabled. Use Runtime > Change runtime type > GPU.")
except Exception as exc:
    DEVICE = "cpu"
    print("Could not import torch, defaulting to CPU:", exc)

DEVICE

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


'cuda'

In [ ]:
EARLY_CYCLE = 5
FIXED_LEN = 60
HORIZON = 50
EPOCHS = 10
BATCH_SIZE = 128
TARGET_MODE = "delta"
FEATURE_MODE = "practical"
SEED = 42
SPLIT_MODE = "condition-gap-within-file"
SPLIT_GAP = 5
EVAL_DOMAIN = ""  # Only used when SPLIT_MODE = "same-domain-eval".

MODELS = [
    "persistence",
    "cpmlp",
    "cpmlp_gru_fusion",
    "cpmlp_gru_residual",
    "cpgru",
    "cpmlp_cpgru_fusion",
    "cpdsconv",
    "cpmlp_cpdsconv_fusion",
    "gru_dsconv",
    "cpmlp_dsconv_fusion",
]

split_label = SPLIT_MODE.replace("-", "_")
if EVAL_DOMAIN and SPLIT_MODE == "same-domain-eval":
    split_label = f"{split_label}_{EVAL_DOMAIN.lower()}"
if SPLIT_MODE == "condition-gap-within-file":
    split_label = f"{split_label}_gap{SPLIT_GAP}"
OUTPUT_DIR = f"comparison_outputs_delta_{split_label}_e{EPOCHS}_h{HORIZON}"
print("models:", ",".join(MODELS))
print("split mode:", SPLIT_MODE)
print("eval domain:", EVAL_DOMAIN or "not used")
print("split gap:", SPLIT_GAP)
print("output:", PROJECT_DIR / OUTPUT_DIR)

## Run Experiment

The residual model pretrains a CPMLP base for `EPOCHS` and then trains the residual branch for another `EPOCHS`, so it can take longer than the other models.

`SPLIT_MODE = "condition-gap-within-file"` tags every preprocessed `.pkl` file with dataset and experiment-condition labels, then splits samples inside each usable file as train | gap | validation | gap | test by target-cycle order. Files with too few samples after reserving the gap are reported in `split_info.json`.

In Colab, results are also copied to `Google Drive/MyDrive/UROP_results/` after the run.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--fixed-len", str(FIXED_LEN),
    "--early-cycle", str(EARLY_CYCLE),
    "--horizon", str(HORIZON),
    "--epochs", str(EPOCHS),
    "--residual-base-epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--models", ",".join(MODELS),
    "--target-mode", TARGET_MODE,
    "--feature-mode", FEATURE_MODE,
    "--split-mode", SPLIT_MODE,
    "--split-gap", str(SPLIT_GAP),
    "--seed", str(SEED),
    "--device", DEVICE,
]
if EVAL_DOMAIN and SPLIT_MODE == "same-domain-eval":
    cmd.extend(["--eval-domain", EVAL_DOMAIN])

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

if IN_COLAB:
    drive_results_dir = Path("/content/drive/MyDrive/UROP_results")
    drive_results_dir.mkdir(parents=True, exist_ok=True)
    local_output = PROJECT_DIR / OUTPUT_DIR
    drive_output = drive_results_dir / OUTPUT_DIR
    if drive_output.exists():
        shutil.rmtree(drive_output)
    shutil.copytree(local_output, drive_output)
    print("Copied results to:", drive_output)

## Overall Metrics

Validation metrics are shown when `compare_soh_models.py` has written the `val_*` columns. Test metrics remain the plain `RMSE`/`MAE` columns.

In [ ]:
out_dir = PROJECT_DIR / OUTPUT_DIR
metrics = pd.read_csv(out_dir / "model_comparison_metrics.csv")
display(metrics)

test_metric_cols = ["RMSE", "MAE", "MAPE_percent", "R2"]
val_metric_cols = ["val_RMSE", "val_MAE", "val_MAPE_percent", "val_R2"]
metric_format = {
    "RMSE": "{:.6f}",
    "MAE": "{:.6f}",
    "MAPE_percent": "{:.3f}",
    "R2": "{:.4f}",
    "val_RMSE": "{:.6f}",
    "val_MAE": "{:.6f}",
    "val_MAPE_percent": "{:.3f}",
    "val_R2": "{:.4f}",
}

available_val_metric_cols = [col for col in val_metric_cols if col in metrics.columns]
available_test_metric_cols = [col for col in test_metric_cols if col in metrics.columns]

if available_val_metric_cols:
    print("Validation metrics")
    display(metrics.set_index("model")[available_val_metric_cols].style.format(metric_format))
else:
    print("Validation metrics are not in model_comparison_metrics.csv. Re-run the experiment with the current compare_soh_models.py.")

print("Test metrics")
display(metrics.set_index("model")[available_test_metric_cols].style.format(metric_format))

In [ ]:
plot_cols = [col for col in ["val_RMSE", "RMSE", "val_MAE", "MAE"] if col in metrics.columns]
ax = metrics.plot.bar(x="model", y=plot_cols, figsize=(12, 5), rot=35)
ax.set_title("Delta target comparison: validation/test RMSE / MAE")
ax.grid(axis="y", alpha=0.3)

## Improvement Against CPMLP

The test split is loaded from `predictions/`; validation is loaded from `validation_predictions/` when available.

In [ ]:
test_pred_dir = out_dir / "predictions"
val_pred_dir = out_dir / "validation_predictions"

test_preds = {}
for path in sorted(test_pred_dir.glob("*_test_predictions.csv")):
    model = path.name.replace("_test_predictions.csv", "")
    test_preds[model] = pd.read_csv(path)

val_preds = {}
if val_pred_dir.exists():
    for path in sorted(val_pred_dir.glob("*_val_predictions.csv")):
        model = path.name.replace("_val_predictions.csv", "")
        val_preds[model] = pd.read_csv(path)

# Keep downstream cells test-focused by default.
preds = test_preds

ref_model = "cpmlp"


def summarize_against_reference(pred_map, ref_model="cpmlp"):
    rows = []
    if ref_model not in pred_map:
        return pd.DataFrame(rows)
    ref = pred_map[ref_model]
    ref_err = (ref["actual_soh"] - ref["pred_soh"]).abs()
    for model, df in pred_map.items():
        err = (df["actual_soh"] - df["pred_soh"]).abs()
        delta = ref_err - err
        rows.append({
            "model": model,
            "mae": float(err.mean()),
            "mae_delta_vs_cpmlp": float(delta.mean()),
            "median_delta_vs_cpmlp": float(delta.median()),
            "improved_ratio_vs_cpmlp": float((delta > 0).mean()),
        })
    return pd.DataFrame(rows).sort_values("mae")


improvement = summarize_against_reference(test_preds, ref_model)
print("Test prediction improvement vs CPMLP")
display(improvement.style.format({
    "mae": "{:.6f}",
    "mae_delta_vs_cpmlp": "{:.6f}",
    "median_delta_vs_cpmlp": "{:.6f}",
    "improved_ratio_vs_cpmlp": "{:.2%}",
}))

if val_preds:
    val_improvement = summarize_against_reference(val_preds, ref_model)
    print("Validation prediction improvement vs CPMLP")
    display(val_improvement.style.format({
        "mae": "{:.6f}",
        "mae_delta_vs_cpmlp": "{:.6f}",
        "median_delta_vs_cpmlp": "{:.6f}",
        "improved_ratio_vs_cpmlp": "{:.2%}",
    }))
else:
    print("No validation prediction files found. Re-run the experiment with the current compare_soh_models.py.")

## Per-Cell Metrics

Per-cell metrics are reported for test and, when present, validation.

In [ ]:
def compute_cell_metrics(pred_map):
    rows = []
    for model, df in pred_map.items():
        for cell_id, g in df.groupby("cell_id"):
            err = g["actual_soh"] - g["pred_soh"]
            rows.append({
                "cell_id": cell_id,
                "model": model,
                "n": len(g),
                "RMSE": float(np.sqrt(np.mean(err ** 2))),
                "MAE": float(np.mean(np.abs(err))),
                "Bias": float(np.mean(g["pred_soh"] - g["actual_soh"])),
            })
    return pd.DataFrame(rows).sort_values(["cell_id", "RMSE"])


cell_metric_format = {
    "RMSE": "{:.6f}",
    "MAE": "{:.6f}",
    "Bias": "{:.6f}",
}

cell_metrics = compute_cell_metrics(test_preds)
print("Test per-cell metrics")
display(cell_metrics.style.format(cell_metric_format))

if val_preds:
    val_cell_metrics = compute_cell_metrics(val_preds)
    print("Validation per-cell metrics")
    display(val_cell_metrics.style.format(cell_metric_format))

## Delta Prediction Sanity Check

This verifies that restored SOH equals `baseline_soh + pred_delta_soh` for test and validation predictions.

In [ ]:
def summarize_delta_outputs(pred_map):
    delta_rows = []
    for model, df in pred_map.items():
        required = {"baseline_soh", "pred_delta_soh", "actual_delta_soh"}
        if required.issubset(df.columns):
            recon_err = (df["pred_soh"] - (df["baseline_soh"] + df["pred_delta_soh"])).abs().max()
            delta_rows.append({
                "model": model,
                "actual_delta_mean": float(df["actual_delta_soh"].mean()),
                "pred_delta_mean": float(df["pred_delta_soh"].mean()),
                "actual_delta_std": float(df["actual_delta_soh"].std()),
                "pred_delta_std": float(df["pred_delta_soh"].std()),
                "reconstruction_max_abs_err": float(recon_err),
            })
    return pd.DataFrame(delta_rows)


delta_format = {
    "actual_delta_mean": "{:.6f}",
    "pred_delta_mean": "{:.6f}",
    "actual_delta_std": "{:.6f}",
    "pred_delta_std": "{:.6f}",
    "reconstruction_max_abs_err": "{:.2e}",
}

test_delta_summary = summarize_delta_outputs(test_preds)
print("Test delta prediction sanity check")
display(test_delta_summary.style.format(delta_format))

if val_preds:
    val_delta_summary = summarize_delta_outputs(val_preds)
    print("Validation delta prediction sanity check")
    display(val_delta_summary.style.format(delta_format))